# Rerun of the added experiments
Uses only the real ISCX 15-second data. Takes roughly 20-30 minutes.

**Before running:** upload `ISCX_VPN_nonVPN_real_CSVs.zip` and `Complete_Research_Results.zip` to Colab's Files panel, then choose Runtime > Run all. Results are saved in `/content/work/exp`.

In [ ]:
!pip install -q shap lime statsmodels
import os,zipfile
for z,d in [('/content/ISCX_VPN_nonVPN_real_CSVs.zip','/content/work/data'),('/content/Complete_Research_Results.zip','/content/work/res')]:
    os.makedirs(d,exist_ok=True);zipfile.ZipFile(z).extractall(d)
os.makedirs('/content/work/exp',exist_ok=True);os.chdir('/content/work/exp')

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL']='3'; os.environ['TF_ENABLE_ONEDNN_OPTS']='0'
import numpy as np,pandas as pd
from sklearn.model_selection import train_test_split as tts
from sklearn.preprocessing import LabelEncoder,StandardScaler
W='/content/work'
FEATS=['duration','total_fiat','total_biat','min_fiat','min_biat','max_fiat','max_biat','mean_fiat','mean_biat','flowPktsPerSecond','flowBytesPerSecond','min_flowiat','max_flowiat','mean_flowiat','std_flowiat']
def load():
    d=pd.read_csv(f'{W}/data/TimeBasedFeatures-Dataset-15s-VPN.csv')
    X=d.drop(columns='class1').replace([np.inf,-np.inf],np.nan);X=X.fillna(X.median())
    y=LabelEncoder().fit_transform(d.class1)
    a,t,ya,yt=tts(X,y,test_size=.2,random_state=42,stratify=y)
    v,te,yv,yte=tts(t,yt,test_size=.5,random_state=42,stratify=yt)
    s=StandardScaler().fit(a)
    idx=[list(X.columns).index(f) for f in FEATS]
    f=lambda z:s.transform(z)[:,idx].astype('float32')
    return f(a),f(v),f(te),ya,yv,yte


In [ ]:
import sys,time,json
import keras,joblib,json,warnings,shap,tensorflow as tf
warnings.filterwarnings('ignore')
from sklearn.metrics import accuracy_score as A,roc_auc_score as U,confusion_matrix as C
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint as PC
from lime.lime_tabular import LimeTabularExplainer as L
a,v,t,ya,yv,yt=load();T=t[...,None]
O=keras.saving.load_model(f'{W}/res/cnn_model.keras',compile=False);I=keras.saving.load_model(f'{W}/res/improved_cnn.keras',compile=False)
RF=joblib.load(f'{W}/res/random_forest.pkl')
bce=keras.losses.BinaryCrossentropy()
def grad(m,x,y):
    x=tf.constant(x);y=tf.constant(y,dtype=tf.float32)
    with tf.GradientTape() as g:
        g.watch(x);l=bce(y,tf.reshape(m(x,training=False),[-1]))
    return g.gradient(l,x).numpy()
def fgsm(m,x,y,e): return x+e*np.sign(grad(m,x,y))
def pgd(m,x,y,e,n=10):
    z=x.copy();s=2.5*e/n
    for _ in range(n): z=np.clip(z+s*np.sign(grad(m,z,y)),x-e,x+e)
    return z
pr=lambda m,x:m.predict(x,verbose=0,batch_size=512).ravel()
E=[0,.01,.05,.1,.15,.2,.3];R=[]
for e in E:
    for n,m in [('Original',O),('Improved',I)]:
        xa=T if e==0 else fgsm(m,T,yt,e);xp=T if e==0 else pgd(m,T,yt,e)
        p=pr(m,xa)>=.5;q=pr(m,xp)>=.5
        R.append(dict(eps=e,model=n,fgsm=A(yt,p)*100,pgd=A(yt,q)*100,fgsm_vpn_share=p.mean()*100,
            rf_transfer=A(yt,RF.predict(xa[...,0]))*100,zero_grad=float((grad(m,T,yt)==0).mean()*100)))
pd.DataFrame(R).round(2).to_csv('robustness_saved.csv',index=False);print(pd.DataFrame(R).round(2).to_string())
# stats
ph={k:(pr(m,T)>=.5).astype(int) for k,m in [('Original',O),('Improved',I)]};ph['RF']=RF.predict(t)
S={}
for k,h in ph.items():
    c=(h==yt).sum();S[k]=dict(acc=c/len(yt)*100,ci=[x*100 for x in PC(c,len(yt),method='wilson')],cm=C(yt,h).tolist())
for x,y_ in [('RF','Improved'),('RF','Original'),('Improved','Original')]:
    ax=ph[x]==yt;bx=ph[y_]==yt;tab=[[(ax&bx).sum(),(ax&~bx).sum()],[(~ax&bx).sum(),(~ax&~bx).sum()]]
    r=mcnemar(tab,exact=False,correction=True);S[f'{x}_vs_{y_}']=dict(tab=np.array(tab).tolist(),chi2=float(r.statistic),p=float(r.pvalue))
json.dump(S,open('stats.json','w'),indent=1,default=float);print(json.dumps(S,default=float))
# SHAP
np.random.seed(42);bg=a[np.random.choice(len(a),100,replace=False)]
sx=t[:300]
rs=shap.TreeExplainer(RF).shap_values(sx);rs=rs[:,:,1] if rs.ndim==3 else rs[1]
cs=[]
for n,m in [('Original',O),('Improved',I)]:
    e=shap.GradientExplainer(m,bg[...,None]);v_=np.array(e.shap_values(sx[...,None],nsamples=200,rseed=42)).squeeze();cs.append(v_);print(n,v_.shape)
np.savez('shap.npz',rf=rs,orig=cs[0],imp=cs[1],x=sx)
tb=pd.DataFrame(dict(feature=FEATS,RF=np.abs(rs).mean(0),Original=np.abs(cs[0]).mean(0),Improved=np.abs(cs[1]).mean(0)))
for c in ['RF','Original','Improved']: tb[c+'_rank']=tb[c].rank(ascending=False).astype(int)
tb.sort_values('RF',ascending=False).round(5).to_csv('shap_importance.csv',index=False);print(tb.sort_values('RF',ascending=False).round(4).to_string())
print('spearman',tb[['RF','Original','Improved']].corr('spearman').round(3).to_string())
# LIME
ex=L(a,feature_names=FEATS,class_names=['Non-VPN','VPN'],mode='classification',discretize_continuous=True,random_state=42)
fr=lambda z:RF.predict_proba(z)
def fc(m): return lambda z:(lambda p:np.c_[1-p,p])(m.predict(z[...,None],verbose=0,batch_size=4096).ravel())
W_={'RF':fr,'Improved':fc(I)};LA={}
for k,f in W_.items():
    arr=np.zeros((100,15))
    for i in range(100):
        mp=dict(ex.explain_instance(t[i],f,num_features=15,num_samples=2000,labels=(1,)).as_map()[1])
        arr[i]=[mp[j] for j in range(15)]
    LA[k]=arr
np.savez('lime.npz',**LA)
lt=pd.DataFrame(dict(feature=FEATS,RF=np.abs(LA['RF']).mean(0),Improved=np.abs(LA['Improved']).mean(0)));lt.round(5).to_csv('lime_importance.csv',index=False)
print(lt.sort_values('RF',ascending=False).round(4).to_string())
for k,sc in [('RF','RF'),('Improved','Improved')]:
    print(k,'lime vs shap spearman',round(pd.Series(lt[k]).corr(tb[sc],method='spearman'),3),'top5 overlap',len(set(lt.nlargest(5,k).feature)&set(tb.nlargest(5,sc).feature)))
# examples
wrong=[i for i in range(len(yt)) if ph['Improved'][i]!=yt[i] and ph['RF'][i]==yt[i]][0]
out={}
for i in [0,wrong]:
    for k,f in W_.items():
        e=ex.explain_instance(t[i],f,num_features=8,num_samples=5000,labels=(1,))
        out[f'{i}_{k}']=dict(true=int(yt[i]),pred=int(f(t[i:i+1]).argmax()),p_vpn=float(f(t[i:i+1])[0,1]),w=e.as_list(label=1))
json.dump(out,open('lime_examples.json','w'),indent=1);print(json.dumps(out)[:1800])
print('feature std (train scaled=1), raw check done')


In [ ]:
import sys,time,json
import keras,joblib,warnings;warnings.filterwarnings('ignore')
from sklearn.metrics import accuracy_score as A
a,v,t,ya,yv,yt=load();T=t[...,None]
O=keras.saving.load_model(f'{W}/res/cnn_model.keras',compile=False);I=keras.saving.load_model(f'{W}/res/improved_cnn.keras',compile=False);RF=joblib.load(f'{W}/res/random_forest.pkl')
R=[];g=np.random.default_rng(42)
for e in [.01,.05,.1,.15,.2,.3]:
    r=[]
    for _ in range(5):
        n=e*g.choice([-1.,1.],size=t.shape).astype('float32')
        r.append([A(yt,RF.predict(t+n))*100,A(yt,O.predict((t+n)[...,None],verbose=0,batch_size=512).ravel()>=.5)*100,A(yt,I.predict((t+n)[...,None],verbose=0,batch_size=512).ravel()>=.5)*100])
    R.append([e]+list(np.mean(r,0).round(2)))
c=pd.DataFrame(R,columns=['eps','RF','Original','Improved']);c.to_csv('random_noise.csv',index=False);print(c.to_string())
d=pd.read_csv(f'{W}/data/TimeBasedFeatures-Dataset-15s-VPN.csv')
tr,_=tts(d,test_size=.2,random_state=42,stratify=d.class1)
q=pd.DataFrame({'std':tr[FEATS].std(),'median_abs':tr[FEATS].abs().median(),'iqr':tr[FEATS].quantile(.75)-tr[FEATS].quantile(.25)})
q['eps.01_over_median']=.01*q['std']/q.median_abs;q['iqr_in_std']=q.iqr/q['std'];print(q.round(3).to_string())
print('frac test within .01 of feature mode etc: median of |z|',np.round(np.median(np.abs(t),axis=0),3))


In [ ]:
import sys,time,json
import keras,tensorflow as tf
from keras import layers as Y
from keras.regularizers import l2
from sklearn.metrics import accuracy_score as A,roc_auc_score as U
from sklearn.ensemble import RandomForestClassifier as RFC
a,v,t,ya,yv,yt=load();Ta,Tv,Tt=a[...,None],v[...,None],t[...,None]
E=[0,.01,.05,.1,.15,.2,.3]
def orig():
    return keras.Sequential([keras.Input((15,1)),Y.Conv1D(64,3,activation='relu'),Y.BatchNormalization(),Y.MaxPooling1D(2),Y.Conv1D(128,3,activation='relu'),Y.BatchNormalization(),Y.Flatten(),Y.Dense(64,activation='relu'),Y.Dropout(.3),Y.Dense(1,activation='sigmoid')])
def imp():
    r=lambda:l2(.001)
    return keras.Sequential([keras.Input((15,1)),Y.Conv1D(32,3,padding='same',activation='relu',kernel_regularizer=r()),Y.BatchNormalization(),Y.Conv1D(64,3,padding='same',activation='relu',kernel_regularizer=r()),Y.BatchNormalization(),Y.GlobalMaxPooling1D(),Y.Dense(64,activation='relu'),Y.Dropout(.4),Y.Dense(32,activation='relu'),Y.Dropout(.3),Y.Dense(1,activation='sigmoid')])
bce=keras.losses.BinaryCrossentropy()
def fgsm(m,x,y,e):
    x=tf.constant(x);y=tf.constant(y,dtype=tf.float32)
    with tf.GradientTape() as g:
        g.watch(x);l=bce(y,tf.reshape(m(x,training=False),[-1]))
    return (x+e*tf.sign(g.gradient(l,x))).numpy()
def pgd(m,x,y,e,n=10):
    z=x.copy();s=2.5*e/n
    for _ in range(n): z=np.clip(fgsm(m,z,y,s),x-e,x+e)
    return z
pr=lambda m,x:m.predict(x,verbose=0,batch_size=512).ravel()
def ev(m):
    d=dict(clean=A(yt,pr(m,Tt)>=.5)*100,auc=U(yt,pr(m,Tt)))
    for e in E[1:]:
        d[f'f{e}']=A(yt,pr(m,fgsm(m,Tt,yt,e))>=.5)*100
    for e in (.05,.1,.2): d[f'p{e}']=A(yt,pr(m,pgd(m,Tt,yt,e))>=.5)*100
    return d
def std(mk,ep,pat,rlr,s):
    keras.utils.set_random_seed(s);m=mk();m.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    cb=[keras.callbacks.EarlyStopping(monitor='val_loss',patience=pat,restore_best_weights=True)]
    if rlr:cb.append(keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=3,min_lr=1e-5))
    st=time.time();h=m.fit(Ta,ya,validation_data=(Tv,yv),epochs=ep,batch_size=64,callbacks=cb,verbose=0);return m,time.time()-st,len(h.history['loss'])
def at(s,et,ep=40):
    keras.utils.set_random_seed(s);m=imp();opt=keras.optimizers.Adam();ds=tf.data.Dataset.from_tensor_slices((Ta,ya.astype('float32'))).shuffle(20000,seed=s).batch(64)
    @tf.function
    def step(x,y):
        with tf.GradientTape() as g:
            g.watch(x);l=bce(y,tf.reshape(m(x,training=False),[-1]))
        xa=x+et*tf.sign(g.gradient(l,x))
        with tf.GradientTape() as g:
            lc=bce(y,tf.reshape(m(x,training=True),[-1]));la=bce(y,tf.reshape(m(xa,training=True),[-1]))
            L=.5*lc+.5*la+tf.add_n(m.losses)
        opt.apply(g.gradient(L,m.trainable_variables),m.trainable_variables)
    best,bw,w,wl=1e9,None,0,0;st=time.time();n=0
    for n in range(ep):
        for x,y in ds: step(x,y)
        vv=.5*float(bce(yv,pr(m,Tv)))+.5*float(bce(yv,pr(m,fgsm(m,Tv,yv,et))))
        if vv<best-0: best,bw,w,wl=vv,m.get_weights(),0,0
        else:
            w+=1;wl+=1
            if wl>=3: opt.learning_rate.assign(max(float(opt.learning_rate)*.5,1e-5));wl=0
            if w>=7:break
    m.set_weights(bw);return m,time.time()-st,n+1
rows=[];SE=[1,2,3,4,5]
def rec(n,s,m,tm,ep,extra={}):
    d=dict(model=n,seed=s,time=tm,epochs=ep,**extra,**ev(m));d['val_clean']=A(yv,pr(m,Tv)>=.5)*100
    for e in (.05,.1): d[f'val_f{e}']=A(yv,pr(m,fgsm(m,Tv,yv,e))>=.5)*100
    rows.append(d);pd.DataFrame(rows).round(4).to_csv('seeds.csv',index=False);print(n,s,round(tm),ep,round(d['clean'],2),round(d['f0.1'],2),flush=True)
for s in SE:
    st=time.time();r=RFC(200,max_depth=20,min_samples_leaf=2,class_weight='balanced',random_state=s,n_jobs=-1).fit(a,ya);tm=time.time()-st
    rows.append(dict(model='RF',seed=s,time=tm,clean=A(yt,r.predict(t))*100,auc=U(yt,r.predict_proba(t)[:,1])))
    m,tm,ep=std(orig,30,5,False,s);rec('Original',s,m,tm,ep)
    m,tm,ep=std(imp,40,7,True,s);rec('Improved',s,m,tm,ep)
    for et in (.05,.1):
        m,tm,ep=at(s,et);rec(f'AT{et}',s,m,tm,ep);m.save(f'at_{et}_{s}.keras')
print('DONE')
